In [ ]:
!pip install xml_to_dict
import requests
from xml_to_dict import XMLtoDict
xd = XMLtoDict()
import pandas as pd
import xml.etree.ElementTree as ET


## 공공데이터 포털 API 방식 시도

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv('SEIBRO_SHARE_API_KEY')
if not api_key:
    from getpass import getpass
    api_key = getpass('SEIBRO_SHARE_API_KEY: ')


In [ ]:
# endcoding

# decoding


In [ ]:
# 고려아연의 예탁결제원 고객번호는 1013
issucoCustno = '593'
secnNm = '고려아연'
rgtStdDt = '20240930'

In [ ]:
url

In [ ]:
url = f'https://api.seibro.or.kr/openapi/service/StockSvc/getSafeDpDutyDepoStatusN1?stdDt=20250103&listTpcd=12&ServiceKey={api_key}'
raw = requests.get(url)
data_dict = xd.parse(raw.content.decode('utf-8'))
data_list = data_dict['SeibroAPI']['vector']['data']


In [ ]:
records = []
for item in data_list:
    record = {
        'SHOTN_ISIN': item['result']['SHOTN_ISIN']['@value'],
        'KOR_SECN_NM': item['result']['KOR_SECN_NM']['@value'],
        'ISSUCO_CUSTNO': item['result']['ISSUCO_CUSTNO']['@value']
    }
    records.append(record)
df = pd.DataFrame(records)

## Seibro 홈페이지 직접 크롤링(requests방식)

In [ ]:
url = 'https://seibro.or.kr/websquare/engine/proworks/callServletService.jsp'

headers = {
    'Accept': 'application/xml',
    'Accept-Encoding': 'gzip, deflate, br, zstd',
    'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7,ja;q=0.6',
    'Content-Type': 'application/xml; charset="UTF-8"',
    'Origin': 'https://seibro.or.kr',
    'Referer': 'https://seibro.or.kr/websquare/control.jsp?w2xPath=/IPORTAL/user/company/BIP_CNTS01021V.xml&menuNo=19',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'
}

payload = '''
<reqParam action="termByImptSkedulList" task="ksd.safe.bip.cnts.Company.process.EntrSkedulPTask">
    <MENU_NO value="19"/>
    <ISSUCO_CUSTNO value=""/>
    <DT_TPCD value=""/>
    <FROMDATE value="20250120"/>
    <TODATE value="20250125"/>
    <LIST_TPCD value="12"/>
    <RGT_RACD value=""/>
</reqParam>
'''

response = requests.post(url, headers=headers, data=payload)

if response.status_code == 200:
    print('Successful')
else:
    print(f"Error: {response.status_code}")



In [ ]:
pass  # Raw API data and exceptions may contain credentials.


In [ ]:
# XML 문자열을 파싱 (response.text를 사용한다고 가정)
root = ET.fromstring(response.text)

# 데이터를 저장할 리스트 생성
data_list = []

# 각 데이터 요소에서 정보 추출
for data in root.findall('.//data'):
    result = data.find('result')
    if result is not None:
        item = {}
        for child in result:
            item[child.tag] = child.get('value')
        data_list.append(item)

# 데이터프레임 생성
df = pd.DataFrame(data_list)

# 데이터프레임 출력
df


In [ ]:
df